In [ ]:
import pandas as pd
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
reddit_posts_filepath = '/content/drive/MyDrive/Dubai_Real_Estate_Data/reddit_data/dubai/dubai_full_raw_data.csv'
posts = pd.read_csv(reddit_posts_filepath, low_memory=False)
print(len(posts))

21292


In [ ]:
reddit_comments_filepath = '/content/drive/MyDrive/Dubai_Real_Estate_Data/reddit_data/dubai/dubai_full_raw_comments.csv'
comments = pd.read_csv(reddit_comments_filepath, low_memory=False)
print(len(comments))

271784


Data preprocessing

In [ ]:
# Strip the "t3_" prefix from comments' link_id to match posts' id
comments["post_id"] = comments["link_id"].str.replace("t3_", "", regex=False)

In [ ]:
#conversion of dates from epoch unix utc
posts["date"] = pd.to_datetime(
    posts["created_utc"],
    unit="s",
    utc=True
)

comments["date"] = pd.to_datetime(
    comments["created_utc"],
    unit="s",
    utc=True
)

In [ ]:
#removing "[removed]"
print("posts/comments that contain '[removed]'")
print((posts["selftext"] == "[removed]").sum())
print((comments["body"] == "[removed]").sum())
print("posts/comments that contain '[deleted]'")
print((posts["selftext"] == "[deleted]").sum())
print((comments["body"] == "[deleted]").sum())


posts = posts[
    ~posts["selftext"].fillna("").str.contains(
        r"\[removed\]|\[deleted\]",
        case=False,
        regex=True
    )
].copy()

comments = comments[
    ~comments["body"].fillna("").str.contains(
        r"\[removed\]|\[deleted\]",
        case=False,
        regex=True
    )
].copy()

posts/comments that contain '[removed]'
10382
8449
posts/comments that contain '[deleted]'
76
1865


In [ ]:
#Exact Deduplication

print(f"Posts before deduplication: {len(posts):,}")
print(f"Comments before deduplication: {len(comments):,}")

posts = posts.drop_duplicates(subset=["id"], keep="first").copy()
comments = comments.drop_duplicates(subset=["id"], keep="first").copy()

print(f"Posts after deduplication: {len(posts):,}")
print(f"Comments after deduplication: {len(comments):,}")

Posts before deduplication: 10,834
Comments before deduplication: 261,470
Posts after deduplication: 10,834
Comments after deduplication: 261,470


In [ ]:
#blank rows

# Posts: remove rows where both title and selftext are empty

posts_before = len(posts)
comments_before = len(comments)

posts = posts[
    ~(
        posts["title"].fillna("").str.strip().eq("")
        &
        posts["selftext"].fillna("").str.strip().eq("")
    )
].copy()

In [ ]:
# Comments: remove rows with empty bodies

comments = comments[
    comments["body"].fillna("").str.strip().ne("")
].copy()

posts_after = len(posts)
comments_after = len(comments)

print(f"Posts before : {posts_before:,}")
print(f"Posts after  : {posts_after:,}")
print(f"Comments before : {comments_before:,}")
print(f"Comments after  : {comments_after:,}")

Posts before : 10,834
Posts after  : 10,834
Comments before : 261,470
Comments after  : 261,456


In [ ]:
#pairing posts and comments together
posts_clean = posts[[
    "id", "title", "selftext", "author", "subreddit",
    "score", "upvote_ratio", "num_comments", "date",
    "permalink", "url"
]].rename(columns={"id": "post_id", "author": "post_author", "score": "post_score"})

comments_clean = comments[[
    "id", "post_id", "parent_id", "body", "author", "score",
    "date", "permalink", "controversiality", "is_submitter"
]].rename(columns={"id": "comment_id", "author": "comment_author", "score": "comment_score"})


df = comments_clean.merge(posts_clean, on="post_id", how="inner")

print(f"Posts: {len(posts_clean):,}")
print(f"Comments: {len(comments_clean):,}")
print(f"Matched comment-post pairs: {len(df):,}")
print(f"No matches comments: {len(comments_clean) - len(df):,}")


Posts: 10,834
Comments: 261,456
Matched comment-post pairs: 255,138
No matches comments: 6,318


In [ ]:
posts_clean["post_text"] = (
    posts_clean["title"].fillna("") + " " +
    posts_clean["selftext"].fillna("")
)
#posts_clean[["title", "selftext", "post_text"]].head(10)

In [ ]:
# Folder path
import os
save_path = "/content/drive/MyDrive/Dubai_Real_Estate_Data/Prajwal final reddit"
os.makedirs(save_path, exist_ok=True)

# Save posts
posts_clean.to_csv(
    f"{save_path}/dubai_posts.csv",
    index=False
)

# Save comments
comments_clean.to_csv(
    f"{save_path}/dubai_comments.csv",
    index=False
)
print("All files saved successfully.")

All files saved successfully.


Load cleaned posts and comments

In [ ]:
file_path = "/content/drive/MyDrive/Dubai_Real_Estate_Data/Prajwal final reddit"
posts_clean = pd.read_csv(f"{file_path}/dubai_posts.csv", low_memory=False)
comments_clean = pd.read_csv(f"{file_path}/dubai_comments.csv", low_memory=False)

In [ ]:
gold1 = pd.read_csv("/content/drive/MyDrive/Dubai_Real_Estate_Data/Prajwal final reddit/dubaiposts_gold_sample1.csv")

Key filtering

In [ ]:
import re

# Real estate keywords
real_estate_keywords = [
     # General
    "property", "properties", "real estate", "housing",

    # Property types
    "apartment", "apartments",
    "villa", "villas",
    "townhouse", "townhouses",
    "penthouse", "penthouses",
    "studio", "studios",
    "flat", "flats",

    # Renting
    "rent", "rental", "renting",
    "tenant", "tenants",
    "landlord", "landlords",
    "lease", "leasing",
    "ejari",
    "security deposit",

    # Buying / Selling
    "buy", "buying", "buyer",
    "sell", "selling", "seller",
    "purchase",
    "mortgage",
    "home loan",

    # Investment
    "investment property",
    "property investment",
    "roi",
    "rental yield",
    "capital appreciation",

    # Agents / Brokers
    "broker", "brokers",
    "agent", "agents",
    "real estate agent",

    # Property management
    "service charge",
    "maintenance fee",

    # Off-plan
    "off plan",
    "off-plan",
    "handover",

    # Ownership
    "freehold",
    "ownership",
    "title deed",

    # Major Dubai developers
    "emaar",
    "damac",
    "nakheel",
    "sobha",
    "azizi",
    "ellington",
    "meraas",
    "danube",

    # Popular Dubai communities
    "dubai marina",
    "business bay",
    "downtown",
    "jvc",
    "jumeirah village circle",
    "jlt",
    "jumeirah lake towers",
    "arabian ranches",
    "dubai hills",
    "palm jumeirah",
    "silicon oasis",
    "motor city",
]

import re

# Build regex pattern
pattern = re.compile(
    r"\b(?:"
    + "|".join(re.escape(k) for k in real_estate_keywords)
    + r")\b",
    flags=re.IGNORECASE
)


# Create keyword filter prediction column
gold1["kwf_label"] = (
    gold1["post_text"]
    .fillna("")
    .str.contains(pattern, na=False)
    .astype(int)
)

# Check counts
print(gold1["kwf_label"].value_counts())


kwf_label
0    399
1    101
Name: count, dtype: int64


In [ ]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)

# Print report
print("Classification Report\n")
print(
    classification_report(
        gold1["actual_label"],
        gold1["kwf_label"],
        target_names=[
            "Not Real Estate",
            "Real Estate"
        ]
    )
)

# Confusion matrix
cm = confusion_matrix(
    gold1["actual_label"],
    gold1["kwf_label"]
)

print("\nConfusion Matrix")
print(cm)

# Extract TN, FP, FN, TP
tn, fp, fn, tp = cm.ravel()

# Save results
kwf_results = {
    "Model": "Keyword Filter",
    "Accuracy": accuracy_score(
        gold1["actual_label"],
        gold1["kwf_label"]
    ),
    "Precision": precision_score(
        gold1["actual_label"],
        gold1["kwf_label"]
    ),
    "Recall": recall_score(
        gold1["actual_label"],
        gold1["kwf_label"]
    ),
    "F1": f1_score(
        gold1["actual_label"],
        gold1["kwf_label"]
    ),
    "TP": tp,
    "TN": tn,
    "FP": fp,
    "FN": fn
}

print(kwf_results)

Classification Report

                 precision    recall  f1-score   support

Not Real Estate       0.98      0.84      0.90       467
    Real Estate       0.24      0.73      0.36        33

       accuracy                           0.83       500
      macro avg       0.61      0.78      0.63       500
   weighted avg       0.93      0.83      0.86       500


Confusion Matrix
[[390  77]
 [  9  24]]
{'Model': 'Keyword Filter', 'Accuracy': 0.828, 'Precision': 0.2376237623762376, 'Recall': 0.7272727272727273, 'F1': 0.3582089552238806, 'TP': np.int64(24), 'TN': np.int64(390), 'FP': np.int64(77), 'FN': np.int64(9)}


Gold Standard Evaluation

In [ ]:
gold_sample = posts_clean.sample(
    n=500,
    random_state=42
).copy()

gold_sample["actual_label"] = ""

gold_sample.to_csv(
    "/content/drive/MyDrive/Dubai_Real_Estate_Data/Prajwal final reddit/posts_raw_gold_sample.csv",
    index=False
)

TF-IDF

In [ ]:
# Use of this as an exploratory/verification technique

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS
import pandas as pd

vectorizer = TfidfVectorizer(
    stop_words=list(ENGLISH_STOP_WORDS),
    min_df=5,
    max_df=0.8
)

X = vectorizer.fit_transform(gold1["post_text"])

feature_names = vectorizer.get_feature_names_out()

# Average TF-IDF score across all documents
avg_tfidf = X.mean(axis=0).A1

tfidf_df = pd.DataFrame({
    "term": feature_names,
    "avg_tfidf": avg_tfidf
})

tfidf_df = tfidf_df.sort_values(
    "avg_tfidf",
    ascending=False
)

tfidf_df.head(50)

,term,avg_tfidf
201,dubai,0.062891
679,uae,0.033617
349,know,0.027899
369,like,0.026767
343,just,0.026355
382,looking,0.023945
466,people,0.021877
697,visa,0.020988
426,need,0.019448
278,guys,0.019217


CorEx

In [ ]:
!pip install corextopic

In [ ]:
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS
import re

my_stop_words = set(ENGLISH_STOP_WORDS)

# Keep important real-estate terms
anchors = [
    ['rera', 'ejari', 'dld', 'eviction'],
    ['property', 'rent', 'landlord', 'tenant'],
    ['mortgage', 'escrow', 'handover'],
    ['apartment', 'villa', 'townhouse', 'BHK'],
    ['marina', 'jvc', 'jlt', 'downtown']
]

for topic_group in anchors:
    for word in topic_group:
        my_stop_words.discard(word)

'''
custom_stopwords = {
    "looking",
    "good",
    "hi",
    "thanks",
    "thank",
    "advance",
    "recommendations",
    "recommendation",
    "help",
    "need",
    "want",
    "advice",
    "appreciate"
}

my_stop_words.update(custom_stopwords)
'''

def clean_text(text):
    text = str(text).lower()
    text = re.sub(r'[^a-zA-Z\s]', ' ', text)
    words = text.split()
    words = [w for w in words if w not in my_stop_words]
    return " ".join(words)

gold1["posts_text_clean"] = gold1["post_text"].apply(clean_text)

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer
from corextopic import corextopic as ct

# ==========================================
# Create Document-Term Matrix
# ==========================================

vectorizer = CountVectorizer(
    min_df=5,
    max_df=0.8,
    binary=True
)

doc_word_matrix = vectorizer.fit_transform(
    gold1["posts_text_clean"]
)

words = list(vectorizer.get_feature_names_out())

print(f"Vocabulary size: {len(words):,}")
print(f"Document-Term Matrix Shape: {doc_word_matrix.shape}")

# ==========================================
# Train CorEx
# ==========================================

topic_model = ct.Corex(
    n_hidden=10,
    seed=42
)

topic_model.fit(
    doc_word_matrix,
    words=words,
    anchors=anchors,
    anchor_strength=3
)

# ==========================================
# Display Topics
# ==========================================

topics = topic_model.get_topics()

for topic_n, topic in enumerate(topics):
    print(f"\nTopic #{topic_n}")
    print([word for word, score, _ in topic[:10]])

Vocabulary size: 723
Document-Term Matrix Shape: (500, 723)

Topic #0
['situation', 'ejari', 'visa', 'current', 'months', 'family', 'questions', 'cancelled', 'needed', 'end']

Topic #1
['rent', 'property', 'tenant', 'landlord', 'advice', 'appreciate', 'advance', 'aed', 'money', 'help']

Topic #2
['apartment', 'long', 'wanted', 'times', 'term', 'able', 'right', 'instead', 'company', 'time']

Topic #3
['marina', 'said', 'told', 'took', 'days', 'process', 'abu', 'dhabi', 'complaint', 'got']

Topic #4
['love', 'hear', 'free', 'old', 'security', 'building', 'll', 'group', 'trip', 'thought']

Topic #5
['year', 'experience', 'job', 'possible', 'month', 'based', 'currently', 'moving', 'recently', 'home']

Topic #6
['ve', 'feel', 'just', 'really', 'offer', 'like', 'especially', 'dubai', 'real', 'high']

Topic #7
['account', 'payment', 'transfer', 'bank', 'issue', 'receive', 'app', 'card', 'send', 'posts']

Topic #8
['good', 'looking', 'people', 'recommendations', 'hi', 'far', 'experiences', 'pr

In [ ]:
topic_assignments = topic_model.labels.argmax(axis=1)

gold1["topic"] = topic_assignments

gold1["topic"].value_counts().sort_index()

,count
topic,
0,224
1,26
2,43
3,34
4,42
5,52
6,21
7,12
8,29


In [ ]:
import pandas as pd

topic_counts = pd.DataFrame(
    topic_model.labels,
    columns=[f"Topic_{i}" for i in range(topic_model.labels.shape[1])]
)

topic_counts.sum().sort_values(ascending=False)

,0
Topic_9,172
Topic_5,165
Topic_6,145
Topic_8,143
Topic_2,104
Topic_4,94
Topic_0,86
Topic_3,84
Topic_7,81
Topic_1,60


In [ ]:
real_estate_topics = [0, 1, 2, 3]

doc_topics = topic_model.labels

gold1["corex_label"] = [
    int(any(row[t] == 1 for t in real_estate_topics))
    for row in doc_topics
]

gold1["corex_label"].value_counts()

,count
corex_label,
0,311
1,189


In [ ]:
from sklearn.metrics import (
    classification_report,
    confusion_matrix
)

print(
    classification_report(
        gold1["actual_label"],
        gold1["corex_label"],
        target_names=[
            "Not Real Estate",
            "Real Estate"
        ]
    )
)

print(
    confusion_matrix(
        gold1["actual_label"],
        gold1["corex_label"]
    )
)

                 precision    recall  f1-score   support

Not Real Estate       0.96      0.64      0.77       467
    Real Estate       0.11      0.64      0.19        33

       accuracy                           0.64       500
      macro avg       0.54      0.64      0.48       500
   weighted avg       0.91      0.64      0.73       500

[[299 168]
 [ 12  21]]


In [ ]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)

cm = confusion_matrix(
    gold1["actual_label"],
    gold1["corex_label"]
)

tn, fp, fn, tp = cm.ravel()

corex_results = {
    "Model": "CorEx",
    "Accuracy": accuracy_score(
        gold1["actual_label"],
        gold1["corex_label"]
    ),
    "Precision": precision_score(
        gold1["actual_label"],
        gold1["corex_label"]
    ),
    "Recall": recall_score(
        gold1["actual_label"],
        gold1["corex_label"]
    ),
    "F1": f1_score(
        gold1["actual_label"],
        gold1["corex_label"]
    ),
    "TP": tp,
    "TN": tn,
    "FP": fp,
    "FN": fn
}

print(corex_results)

{'Model': 'CorEx', 'Accuracy': 0.64, 'Precision': 0.1111111111111111, 'Recall': 0.6363636363636364, 'F1': 0.1891891891891892, 'TP': np.int64(21), 'TN': np.int64(299), 'FP': np.int64(168), 'FN': np.int64(12)}


BERTopic

In [ ]:
!pip install bertopic

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.7/154.7 kB 5.6 MB/s eta 0:00:00


In [ ]:
docs = gold1["posts_text_clean"].astype(str).tolist()

In [ ]:
from bertopic import BERTopic

topic_model = BERTopic(
    min_topic_size=10,
    calculate_probabilities=True,
    verbose=True
)

topics, probs = topic_model.fit_transform(docs)

2026-06-23 01:26:51,189 - BERTopic - Embedding - Transforming documents to embeddings.


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/16 [00:00<?, ?it/s]

2026-06-23 01:26:58,302 - BERTopic - Embedding - Completed ✓
2026-06-23 01:26:58,303 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2026-06-23 01:27:09,284 - BERTopic - Dimensionality - Completed ✓
2026-06-23 01:27:09,285 - BERTopic - Cluster - Start clustering the reduced embeddings
2026-06-23 01:27:09,323 - BERTopic - Cluster - Completed ✓
2026-06-23 01:27:09,327 - BERTopic - Representation - Fine-tuning topics using representation models.
2026-06-23 01:27:09,372 - BERTopic - Representation - Completed ✓


In [ ]:
topic_info = topic_model.get_topic_info()

display(topic_info)

for topic_id in topic_info.Topic:
    if topic_id != -1:
        print(f"\nTopic {topic_id}")
        print(topic_model.get_topic(topic_id))

,Topic,Count,Name,Representation,Representative_Docs
0,-1,230,-1_dubai_visa_uae_just,"[dubai, visa, uae, just, know, people, aed, he...",[need honest advice bhk purchase dubai golf te...
1,0,59,0_dubai_like_day_trip,"[dubai, like, day, trip, places, shops, sugges...",[time dubai m planning trip dubai march like k...
2,1,44,1_ve_area_building_noise,"[ve, area, building, noise, park, reach, today...",[people leaving hey guys seeing posts facebook...
3,2,34,2_bank_amazon_aed_card,"[bank, amazon, aed, card, transfer, usd, just,...",[amazon chat support passed like ball got tran...
4,3,31,3_people_just_meet_looking,"[people, just, meet, looking, like, know, join...",[north indians al furjan meet hey fun loving e...
5,4,21,4_job_year_dubai_schools,"[job, year, dubai, schools, cfa, planning, fam...",[big career dubai vs staying saudi family move...
6,5,21,5_https_com_www_uae,"[https, com, www, uae, ae, alerts, emergency, ...",[attacks megathread megathread posts related o...
7,6,14,6_termination_terminated_notice_company,"[termination, terminated, notice, company, lea...",[terminated probationary period employee right...
8,7,12,7_gym_study_dip_access,"[gym, study, dip, access, area, looking, room,...",[need free study coworking spaces dso friend w...
9,8,12,8_visa_uae_months_saudi,"[visa, uae, months, saudi, virtual, travel, in...",[t saudi visa months uae residency resident sa...



Topic 0
[('dubai', np.float64(0.08680137583276501)), ('like', np.float64(0.03262904790387144)), ('day', np.float64(0.03140595199971107)), ('trip', np.float64(0.030497255355464215)), ('places', np.float64(0.02966601573906847)), ('shops', np.float64(0.029041707450106067)), ('suggestions', np.float64(0.028634297270862398)), ('hotel', np.float64(0.02772477759587656)), ('ve', np.float64(0.02688057668838052)), ('dhabi', np.float64(0.026650814371894087))]

Topic 1
[('ve', np.float64(0.0444212919850356)), ('area', np.float64(0.04366343664296881)), ('building', np.float64(0.0407796932909209)), ('noise', np.float64(0.040448023029741625)), ('park', np.float64(0.03606816693097822)), ('reach', np.float64(0.03486390549729375)), ('today', np.float64(0.03425251990826234)), ('like', np.float64(0.033182082614106545)), ('mamzar', np.float64(0.03277741981841988)), ('just', np.float64(0.030674295568491914))]

Topic 2
[('bank', np.float64(0.05611861171650708)), ('amazon', np.float64(0.03984340328117712)), 

In [ ]:
real_estate_topics_bert = [1, 10] #expand these topics and include more

gold1["bertopic_topic"] = topics

gold1["bertopic_label"] = (
    gold1["bertopic_topic"]
    .isin(real_estate_topics_bert)
    .astype(int)
)

gold1["bertopic_label"].value_counts()

,count
bertopic_label,
0,445
1,55


In [ ]:
from sklearn.metrics import (
    classification_report,
    confusion_matrix
)

print(
    classification_report(
        gold1["actual_label"],
        gold1["bertopic_label"],
        target_names=[
            "Not Real Estate",
            "Real Estate"
        ]
    )
)

print(
    confusion_matrix(
        gold1["actual_label"],
        gold1["bertopic_label"]
    )
)

                 precision    recall  f1-score   support

Not Real Estate       0.96      0.91      0.93       467
    Real Estate       0.24      0.39      0.30        33

       accuracy                           0.88       500
      macro avg       0.60      0.65      0.61       500
   weighted avg       0.91      0.88      0.89       500

[[425  42]
 [ 20  13]]


In [ ]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)

cm = confusion_matrix(
    gold1["actual_label"],
    gold1["bertopic_label"]
)

tn, fp, fn, tp = cm.ravel()

bertopic_results = {
    "Model": "BERTopic",
    "Accuracy": accuracy_score(
        gold1["actual_label"],
        gold1["bertopic_label"]
    ),
    "Precision": precision_score(
        gold1["actual_label"],
        gold1["bertopic_label"]
    ),
    "Recall": recall_score(
        gold1["actual_label"],
        gold1["bertopic_label"]
    ),
    "F1": f1_score(
        gold1["actual_label"],
        gold1["bertopic_label"]
    ),
    "TP": tp,
    "TN": tn,
    "FP": fp,
    "FN": fn
}

Semantic Similarity Filtering with cosine similarity

BART-MNLI

In [ ]:
!pip install transformers torch

In [ ]:
from transformers import pipeline

classifier_bart = pipeline(
    "zero-shot-classification",
    model="facebook/bart-large-mnli",
    device=0  # remove if not using GPU
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:124: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/1.15k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.63G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/515 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

In [ ]:
candidate_labels = [
    "real estate",
    "not real estate"
]

text = """
I am looking to buy a villa in Dubai Hills.
Any recommendations for good developers?
"""

result = classifier_bart(
    text,
    candidate_labels
)

print(result)

{'sequence': '\nI am looking to buy a villa in Dubai Hills.\nAny recommendations for good developers?\n', 'labels': ['real estate', 'not real estate'], 'scores': [0.9722742438316345, 0.02772580273449421]}


In [ ]:
result["scores"]

[0.9722742438316345, 0.02772580273449421]

In [ ]:
predictions = []

for text in gold1["post_text"].fillna(""):

    result = classifier_bart(
        text[:1024],
        candidate_labels
    )

    predicted_label = result["labels"][0]

    predictions.append(
        1 if predicted_label == "real estate" else 0
    )

gold1["bart_prediction"] = predictions

In [ ]:
from sklearn.metrics import (
    classification_report,
    confusion_matrix
)

print(
    classification_report(
        gold1["actual_label"],
        gold1["bart_prediction"]
    )
)

print(
    confusion_matrix(
        gold1["actual_label"],
        gold1["bart_prediction"]
    )
)

              precision    recall  f1-score   support

           0       0.98      0.90      0.94       467
           1       0.34      0.76      0.47        33

    accuracy                           0.89       500
   macro avg       0.66      0.83      0.70       500
weighted avg       0.94      0.89      0.91       500

[[418  49]
 [  8  25]]


In [ ]:
scores = []

for text in gold1["post_text"].fillna(""):
    result = classifier_bart(
        text[:1024],
        candidate_labels
    )

    re_score = result["scores"][
        result["labels"].index("real estate")
    ]

    scores.append(re_score)

pd.Series(scores).describe()

,0
count,500.000000
mean,0.350015
std,0.202796
min,0.055277
25%,0.209628
50%,0.305976
75%,0.432694
max,0.996554


In [ ]:
gold1["bart_prediction"].value_counts()

,count
bart_prediction,
0,426
1,74


In [ ]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)

cm = confusion_matrix(
    gold1["actual_label"],
    gold1["bart_label"]
)

tn, fp, fn, tp = cm.ravel()

bart_results = {
    "Model": "BART-MNLI",
    "Accuracy": accuracy_score(
        gold1["actual_label"],
        gold1["bart_label"]
    ),
    "Precision": precision_score(
        gold1["actual_label"],
        gold1["bart_label"]
    ),
    "Recall": recall_score(
        gold1["actual_label"],
        gold1["bart_label"]
    ),
    "F1": f1_score(
        gold1["actual_label"],
        gold1["bart_label"]
    ),
    "TP": tp,
    "TN": tn,
    "FP": fp,
    "FN": fn
}

DeBERTa NLI

In [ ]:
!pip install transformers sentencepiece

In [ ]:
from transformers import pipeline

classifier_deberta = pipeline(
    "zero-shot-classification",
    model="MoritzLaurer/deberta-v3-large-zeroshot-v2.0",
    device=0
)

config.json:   0%|          | 0.00/1.02k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/870M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/394 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/1.26k [00:00<?, ?B/s]

spm.model:   0%|          | 0.00/2.46M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/8.66M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/23.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/970 [00:00<?, ?B/s]

In [ ]:
text = """
I am looking to buy a villa in Dubai Hills.
Any recommendations for good developers?
"""

candidate_labels = [
    "real estate",
    "not real estate"
]

result = classifier_deberta(
    text,
    candidate_labels
)

print(result)

{'sequence': '\nI am looking to buy a villa in Dubai Hills.\nAny recommendations for good developers?\n', 'labels': ['real estate', 'not real estate'], 'scores': [0.9991800785064697, 0.0008199320873245597]}


In [ ]:
import pandas as pd

predictions = []

for text in gold1["post_text"].fillna(""):

    result = classifier_deberta(
        text[:1024],
        candidate_labels
    )

    predictions.append(result["labels"][0])

gold1["deberta_prediction"] = predictions

gold1["deberta_label"] = (
    gold1["deberta_prediction"] == "real estate"
).astype(int)

In [ ]:
from sklearn.metrics import (
    classification_report,
    confusion_matrix
)

print(
    classification_report(
        gold1["actual_label"],
        gold1["deberta_label"]
    )
)

print(
    confusion_matrix(
        gold1["actual_label"],
        gold1["deberta_label"]
    )
)

              precision    recall  f1-score   support

           0       0.98      0.99      0.98       467
           1       0.77      0.70      0.73        33

    accuracy                           0.97       500
   macro avg       0.87      0.84      0.86       500
weighted avg       0.96      0.97      0.97       500

[[460   7]
 [ 10  23]]


In [ ]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)

cm = confusion_matrix(
    gold1["actual_label"],
    gold1["deberta_label"]
)

tn, fp, fn, tp = cm.ravel()

deberta_results = {
    "Model": "DeBERTa-NLI",
    "Accuracy": accuracy_score(
        gold1["actual_label"],
        gold1["deberta_label"]
    ),
    "Precision": precision_score(
        gold1["actual_label"],
        gold1["deberta_label"]
    ),
    "Recall": recall_score(
        gold1["actual_label"],
        gold1["deberta_label"]
    ),
    "F1": f1_score(
        gold1["actual_label"],
        gold1["deberta_label"]
    ),
    "TP": tp,
    "TN": tn,
    "FP": fp,
    "FN": fn
}

COMPARISON

In [ ]:
comparison = pd.DataFrame([
    kwf_results, #this acts like a baseline not used for final dataset
    #corex_results, #drop this as this is an unsupervised model and requires intervention
    bertopic_results, #exploratory analysis techinque
    bart_results,
    deberta_results
]) #it is between BART and DeBERTa justified based on threshold values.

comparison = comparison.sort_values(
    "F1",
    ascending=False
)

comparison

,Model,Accuracy,Precision,Recall,F1,TP,TN,FP,FN
3,DeBERTa-NLI,0.966,0.766667,0.696970,0.730159,23,460,7,10
0,Keyword Filter,0.828,0.237624,0.727273,0.358209,24,390,77,9
1,BERTopic,0.876,0.236364,0.393939,0.295455,13,425,42,20
2,BART-MNLI,0.934,0.000000,0.000000,0.000000,0,467,0,33


In [ ]:
gold1.head(5)

,post_id,title,selftext,post_author,subreddit,post_score,upvote_ratio,num_comments,date,permalink,...,kwf_label,posts_text_clean,topic,corex_label,bertopic_topic,bertopic_label,bart_prediction,bart_label,deberta_prediction,deberta_label
0,1sgkjpb,"Think what you will about Parkin, but this is ...",Honestly Salik should do the same.,urmomma123,dubai,1,0.56,1,2026-04-09 09:15:12+00:00,/r/dubai/comments/1sgkjpb/think_what_you_will_...,...,0,think parkin interesting initiative honestly s...,0,0,1,1,0,0,not real estate,0
1,1rbpaa4,Best KFC branch?,Not all KFCs are equal… which branch is the best?,DonElios,dubai,1,0.67,0,2026-02-22 15:53:40+00:00,/r/dubai/comments/1rbpaa4/best_kfc_branch/,...,0,best kfc branch kfcs equal branch best,0,0,-1,0,0,0,not real estate,0
2,1rv7y6h,Are they still firing the Iftar Cannon?,I wanted to see for some time now but given th...,John21st,dubai,4,0.61,9,2026-03-16 12:23:24+00:00,/r/dubai/comments/1rv7y6h/are_they_still_firin...,...,0,firing iftar cannon wanted time given situatio...,0,0,-1,0,0,0,not real estate,0
3,1sd429o,Man with van 3 tonne and 1 tonne,Hi would love a recommendation for family busi...,Rustallo,dubai,1,1.00,0,2026-04-05 13:47:05+00:00,/r/dubai/comments/1sd429o/man_with_van_3_tonne...,...,0,man van tonne tonne hi love recommendation fam...,4,0,-1,0,0,0,not real estate,0
4,1s3y41x,Iranian hospital closed any suggestions on equ...,"Hi everyone,\n\nI’m looking for recommendation...",Willing_Temporary_69,dubai,32,0.90,11,2026-03-26 04:50:28+00:00,/r/dubai/comments/1s3y41x/iranian_hospital_clo...,...,0,iranian hospital closed suggestions equaly aff...,5,0,-1,0,0,0,not real estate,0
